# Proyecto Integrador - Módulo de Streaming en Tiempo Real
**Universidad Santo Tomás - Seccional Tunja**
**Materia:** BIG DATA 2026-2
**E-commerce:** Brazilian E-Commerce Public Dataset by Olist (98,666 órdenes reales)
**Temática:** Simulación de Flujo de Órdenes y Monitoreo de Métricas en Vivo con PySpark Structured Streaming

### 1. Configuración del Entorno Spark y Variables de Sistema

In [1]:
import os
import shutil
import time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import col, sum as spark_sum, count as spark_count, round as spark_round

# Configurar variables de Hadoop para Windows
os.environ['HADOOP_HOME'] = 'C:/hadoop'
os.environ['PATH'] = 'C:/hadoop/bin;' + os.environ.get('PATH', '')

# Crear sesión de Spark
spark = SparkSession.builder \
    .appName('Olist_Streaming_Module') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '2') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Sesión de PySpark iniciada correctamente.')

c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Sesión de PySpark iniciada correctamente.


### 2. Preparación de Directorios y Partición del Dataset

In [2]:
base_dir = 'C:/Users/Asus/.gemini/antigravity/scratch/proyecto-integrador-bigdata'
stream_input = os.path.join(base_dir, 'stream_input')
checkpoint_dir = os.path.join(base_dir, 'stream_checkpoint')

if os.path.exists(stream_input):
    shutil.rmtree(stream_input)
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)

os.makedirs(stream_input, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

csv_path = os.path.join(base_dir, 'data', 'olist_consolidado.csv')
df_full = pd.read_csv(csv_path)

cols_stream = ['order_id', 'order_status', 'customer_state', 'payment_type', 'total_order_value', 'product_category']
df_sample = df_full[cols_stream].dropna()

# Generamos 3 lotes de órdenes
lote1 = df_sample.iloc[0:2000]
lote2 = df_sample.iloc[2000:5000]
lote3 = df_sample.iloc[5000:10000]

# Guardamos Lote 1 inicial
lote1.to_csv(os.path.join(stream_input, 'lote_1.csv'), index=False)
print(f'Lote 1 guardado: {len(lote1)} órdenes listas para ser consumidas.')

Lote 1 guardado: 2000 órdenes listas para ser consumidas.


### 3. Definición de Esquema y Configuración del readStream

In [3]:
schema = StructType([
    StructField('order_id', StringType(), True),
    StructField('order_status', StringType(), True),
    StructField('customer_state', StringType(), True),
    StructField('payment_type', StringType(), True),
    StructField('total_order_value', DoubleType(), True),
    StructField('product_category', StringType(), True)
])

df_stream = spark.readStream \
    .option('header', 'true') \
    .schema(schema) \
    .csv(stream_input)

print('Esquema del flujo de streaming configurado:')
df_stream.printSchema()

Esquema del flujo de streaming configurado:
root
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- total_order_value: double (nullable = true)
 |-- product_category: string (nullable = true)



### 4. Definición de la Métrica en Vivo y Arranque del Stream

In [4]:
metrica_estados = df_stream.groupBy('customer_state') \
    .agg(
        spark_count('order_id').alias('total_ordenes'),
        spark_round(spark_sum('total_order_value'), 2).alias('ventas_totales')
    )

query = metrica_estados.writeStream \
    .outputMode('complete') \
    .format('memory') \
    .queryName('metricas_en_vivo') \
    .option('checkpointLocation', checkpoint_dir) \
    .start()

print('Stream iniciado. Esperando procesamiento del primer micro-batch...')
time.sleep(4)

print('>>> Estado inicial de la métrica tras Lote 1 (Top 5 estados):')
spark.sql('SELECT * FROM metricas_en_vivo ORDER BY ventas_totales DESC LIMIT 5').show()

Stream iniciado. Esperando procesamiento del primer micro-batch...
>>> Estado inicial de la métrica tras Lote 1 (Top 5 estados):
+--------------+-------------+--------------+
|customer_state|total_ordenes|ventas_totales|
+--------------+-------------+--------------+
+--------------+-------------+--------------+



### 5. Simulación de Llegada de Datos y Actualización Dinámica

In [5]:
# Simulación de Lote 2
print('Simulando llegada de Lote 2 (3,000 nuevas órdenes)...')
lote2.to_csv(os.path.join(stream_input, 'lote_2.csv'), index=False)
time.sleep(5)

print('>>> MÉTRICA EN VIVO ACTUALIZADA TRAS LOTE 2 (5,000 órdenes acumuladas):')
spark.sql('SELECT * FROM metricas_en_vivo ORDER BY ventas_totales DESC LIMIT 5').show()

# Simulación de Lote 3
print('Simulando llegada de Lote 3 (5,000 nuevas órdenes)...')
lote3.to_csv(os.path.join(stream_input, 'lote_3.csv'), index=False)
time.sleep(5)

print('>>> MÉTRICA EN VIVO ACTUALIZADA TRAS LOTE 3 (10,000 órdenes acumuladas):')
spark.sql('SELECT * FROM metricas_en_vivo ORDER BY ventas_totales DESC LIMIT 5').show()

Simulando llegada de Lote 2 (3,000 nuevas órdenes)...
>>> MÉTRICA EN VIVO ACTUALIZADA TRAS LOTE 2 (5,000 órdenes acumuladas):
+--------------+-------------+--------------+
|customer_state|total_ordenes|ventas_totales|
+--------------+-------------+--------------+
|            SP|          856|     117498.08|
|            RJ|          246|      48678.68|
|            MG|          217|      32871.48|
|            RS|          116|      19518.74|
|            PR|          114|      17616.71|
+--------------+-------------+--------------+

Simulando llegada de Lote 3 (5,000 nuevas órdenes)...
>>> MÉTRICA EN VIVO ACTUALIZADA TRAS LOTE 3 (10,000 órdenes acumuladas):
+--------------+-------------+--------------+
|customer_state|total_ordenes|ventas_totales|
+--------------+-------------+--------------+
|            SP|         4233|     608506.28|
|            RJ|         1334|     221208.25|
|            MG|         1119|     174412.68|
|            PR|          518|      93034.15|
|         

### 6. Cierre del Flujo de Streaming y Detención de la Sesión

In [6]:
query.stop()
spark.stop()
print('Stream detenido y sesión de Spark cerrada correctamente.')

Stream detenido y sesión de Spark cerrada correctamente.
